In [0]:

dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# Fixed 2026-11-26: catalog adb_caresync does not exist; the landing volume is databrickscaresync.lamding.landingvolume
base_landing_path = "/Volumes/databrickscaresync/lamding/landingvolume"
source_path = f"{base_landing_path}/{source_system}"
table_path = f"{source_path}/{table_name}"
landing_path = f"{table_path}/{landing_timestamp}/"


def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


missing_path = next((path for path in [source_path, table_path, landing_path] if not path_exists(path)), None)

if missing_path:
    message = (
        f"No new files are received for source_system={source_system}, "
        f"table_name={table_name}, landing_timestamp={landing_timestamp}."
    )
    print(message)
    dbutils.notebook.exit(message)

     

In [0]:
# Fixed 2026-11-26: this cell previously used `landingdf` before any read had happened.
# Read the landing files first; schema guard, metadata and write follow in the next cells.
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

print(f"[LANDING PATH] {landing_path}")
print(f"[LANDING COLUMNS] {landing_df.columns}")

In [0]:
import re

# Required business columns per table (metadata columns are added later).
# Keep in sync with the copy in bronzetosilver.
TABLE_CONTRACTS = {
    "patients": ["patient_id", "first_name", "last_name", "date_of_birth", "gender", "city", "state",
                 "registered_hospital_id", "insurance_provider_id"],
    "hospitals": ["hospital_id", "hospital_name", "city", "state", "hospital_type", "bed_count"],
    "doctors": ["doctor_id", "first_name", "last_name", "specialty", "hospital_id", "phone_number"],
    "insurance_providers": ["insurance_provider_id", "provider_name", "provider_type", "contact_email",
                            "coverage_region"],
    # Per STTM sttm_silver_to_gold.xlsx
    "lab_results": ["lab_result_id", "patient_id", "hospital_id", "doctor_id", "test_name", "test_date",
                    "result_status"],
}

if not re.fullmatch(r"[a-z0-9_]+", table_name or ""):
    raise ValueError(f"[SCHEMA GUARD] Invalid table_name: '{table_name}'")
if table_name not in TABLE_CONTRACTS:
    raise ValueError(f"[SCHEMA GUARD] No column contract for table '{table_name}'. "
                     "Add it to TABLE_CONTRACTS before loading. Nothing written to Bronze.")

actual_cols = {c.lower() for c in landing_df.columns}
missing = [c for c in TABLE_CONTRACTS[table_name] if c not in actual_cols]
if missing:
    looks_like = [t for t, cols in TABLE_CONTRACTS.items() if t != table_name and set(cols) <= actual_cols]
    hint = (f" The files match the {looks_like} contract - the extract probably copied the wrong source table."
            if looks_like else "")
    raise ValueError(
        f"[SCHEMA GUARD] Landing files for '{table_name}' are missing {missing}. "
        f"Actual columns: {sorted(actual_cols)}.{hint} Nothing written to Bronze."
    )

landing_count = landing_df.count()
if landing_count == 0:
    dbutils.notebook.exit("0")

print(f"[SCHEMA GUARD PASSED] {table_name}: {landing_count} rows, all required columns present")
     

In [0]:

from pyspark.sql import functions as F

# Bronze tables carry landing_timestamp + ingestion_timestamp (not insert_timestamp)
updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
    .withColumn("ingestion_timestamp", F.current_timestamp())

In [0]:
# Restored 2026-11-26: the write step was missing. Append only - no overwrite and no
# schema merge, so a table whose existing columns differ fails instead of being changed.
bronze_table = f"databrickscaresync.bronze.{table_name}"
print(f"[WRITE] Appending {landing_count} rows to {bronze_table}")
updated_df.write.mode("append").saveAsTable(bronze_table)
print(f"[WRITE COMPLETE] {bronze_table}")
dbutils.notebook.exit(str(landing_count))